In [1]:
import sys
from pathlib import Path

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agents.investigation_agent.schemas import (
    RiskSignal,
    EvidenceItem,
    InvestigationCase,
    InvestigationResult
)

print("Investigation schemas imported successfully.")

Investigation schemas imported successfully.


In [2]:
aml_signal = RiskSignal(
    source="AML Model",
    signal_type="AML Risk",
    score=0.76,
    level="High",
    description="Transaction exceeds the AML screening threshold."
)

anomaly_signal = RiskSignal(
    source="Anomaly Engine",
    signal_type="Transaction Anomaly",
    level="High",
    description="Transaction was flagged by both Isolation Forest and DBSCAN."
)

case = InvestigationCase(
    case_id="CASE-001",
    customer_id="CUST-001",
    transaction_id="TX-001",
    risk_signals=[
        aml_signal,
        anomaly_signal
    ],
    priority="High"
)

print(case)

InvestigationCase(case_id='CASE-001', customer_id='CUST-001', transaction_id='TX-001', risk_signals=[RiskSignal(source='AML Model', signal_type='AML Risk', score=0.76, level='High', description='Transaction exceeds the AML screening threshold.', metadata={}), RiskSignal(source='Anomaly Engine', signal_type='Transaction Anomaly', score=None, level='High', description='Transaction was flagged by both Isolation Forest and DBSCAN.', metadata={})], evidence=[], priority='High', status='Open')


In [4]:
from agents.investigation_agent.investigator import (
    create_investigation_case
)

test_case = create_investigation_case(
    case_id="CASE-002",
    customer_id="CUST-002",
    transaction_id="TX-002",
    aml_score=0.82,
    aml_level="High",
    anomaly_level="High",
    financial_risk_score=0.67,
    financial_risk_level="High"
)

print(test_case)

InvestigationCase(case_id='CASE-002', customer_id='CUST-002', transaction_id='TX-002', risk_signals=[RiskSignal(source='AML Model', signal_type='AML Risk', score=0.82, level='High', description='AML screening signal for the transaction.', metadata={}), RiskSignal(source='Anomaly Engine', signal_type='Transaction Anomaly', score=None, level='High', description='Unsupervised transaction anomaly signal.', metadata={}), RiskSignal(source='Financial Risk Engine', signal_type='Financial Risk', score=0.67, level='High', description='Customer financial early-warning signal.', metadata={})], evidence=[], priority=None, status='Open')


In [5]:
from agents.investigation_agent.investigator import (
    create_investigation_case
)

test_case = create_investigation_case(
    case_id="CASE-002",
    customer_id="CUST-002",
    transaction_id="TX-002",
    aml_score=0.82,
    aml_level="High",
    anomaly_level="High",
    financial_risk_score=0.67,
    financial_risk_level="High"
)

print("Priority:", test_case.priority)
print(test_case)

Priority: None
InvestigationCase(case_id='CASE-002', customer_id='CUST-002', transaction_id='TX-002', risk_signals=[RiskSignal(source='AML Model', signal_type='AML Risk', score=0.82, level='High', description='AML screening signal for the transaction.', metadata={}), RiskSignal(source='Anomaly Engine', signal_type='Transaction Anomaly', score=None, level='High', description='Unsupervised transaction anomaly signal.', metadata={}), RiskSignal(source='Financial Risk Engine', signal_type='Financial Risk', score=0.67, level='High', description='Customer financial early-warning signal.', metadata={})], evidence=[], priority=None, status='Open')


In [6]:
import importlib
import agents.investigation_agent.investigator as investigator

importlib.reload(investigator)

test_case = investigator.create_investigation_case(
    case_id="CASE-002",
    customer_id="CUST-002",
    transaction_id="TX-002",
    aml_score=0.82,
    aml_level="High",
    anomaly_level="High",
    financial_risk_score=0.67,
    financial_risk_level="High"
)

print("Priority:", test_case.priority)
print(test_case)

Priority: Critical
InvestigationCase(case_id='CASE-002', customer_id='CUST-002', transaction_id='TX-002', risk_signals=[RiskSignal(source='AML Model', signal_type='AML Risk', score=0.82, level='High', description='AML screening signal for the transaction.', metadata={}), RiskSignal(source='Anomaly Engine', signal_type='Transaction Anomaly', score=None, level='High', description='Unsupervised transaction anomaly signal.', metadata={}), RiskSignal(source='Financial Risk Engine', signal_type='Financial Risk', score=0.67, level='High', description='Customer financial early-warning signal.', metadata={})], evidence=[], priority='Critical', status='Open')


In [7]:
import importlib
import agents.investigation_agent.evidence as evidence_module

importlib.reload(evidence_module)

case_with_evidence = evidence_module.attach_signal_evidence(
    test_case
)

print("Evidence count:", len(case_with_evidence.evidence))

for item in case_with_evidence.evidence:
    print(
        f"- {item.source}: {item.description}"
    )

Evidence count: 3
- AML Model: AML Risk score: 0.820
- Anomaly Engine: Transaction Anomaly level: High
- Financial Risk Engine: Financial Risk score: 0.670


In [8]:
def attach_context_evidence(
    case: InvestigationCase,
    context: dict
) -> InvestigationCase:

    for key, value in context.items():
        case.evidence.append(
            EvidenceItem(
                evidence_type="Context",
                description=f"{key}: {value}",
                value=value,
                source="Case Context"
            )
        )

    return case

In [10]:
import importlib
import agents.investigation_agent.evidence as evidence_module

importlib.reload(evidence_module)

context = {
    "TransactionAmount": 1531.42,
    "AccountBalance": 860.11,
    "LoginAttempts": 4,
    "DelayedPaymentMonths": 3
}

case_with_context = evidence_module.attach_context_evidence(
    test_case,
    context
)

print("Total evidence count:", len(case_with_context.evidence))

for item in case_with_context.evidence:
    print(
        f"- [{item.evidence_type}] "
        f"{item.source}: {item.description}"
    )

Total evidence count: 7
- [Model Signal] AML Model: AML Risk score: 0.820
- [Model Signal] Anomaly Engine: Transaction Anomaly level: High
- [Model Signal] Financial Risk Engine: Financial Risk score: 0.670
- [Context] Case Context: TransactionAmount: 1531.42
- [Context] Case Context: AccountBalance: 860.11
- [Context] Case Context: LoginAttempts: 4
- [Context] Case Context: DelayedPaymentMonths: 3


In [11]:
import importlib
import agents.investigation_agent.investigator as investigator

importlib.reload(investigator)

findings = investigator.generate_findings(
    case_with_context
)

print("Generated findings:")
print("-------------------")

for finding in findings:
    print("-", finding)

Generated findings:
-------------------
- High aml risk signal from AML Model.
- High transaction anomaly signal from Anomaly Engine.
- High financial risk signal from Financial Risk Engine.
- Transaction amount exceeds the current account balance.
- Multiple login attempts were associated with the transaction.
- Customer has repeated delayed-payment behaviour.


In [12]:
import importlib
import agents.investigation_agent.investigator as investigator

importlib.reload(investigator)

findings = investigator.generate_findings(
    case_with_context
)

actions = investigator.generate_recommended_actions(
    case_with_context,
    findings
)

print("Case priority:", case_with_context.priority)

print("\nRecommended actions:")
print("--------------------")

for action in actions:
    print("-", action)

Case priority: Critical

Recommended actions:
--------------------
- Escalate the case for immediate analyst review.
- Review recent transaction history for related unusual activity.
- Review authentication and login activity associated with the account.
- Review recent repayment behaviour and financial stress indicators.
- Review AML-related transaction evidence and linked account activity.


In [13]:
import importlib
import agents.investigation_agent.investigator as investigator

importlib.reload(investigator)

result = investigator.build_deterministic_investigation_result(
    case_with_context
)

print("Case ID:", result.case_id)
print("Overall risk level:", result.overall_risk_level)
print("Confidence:", result.confidence)

print("\nSummary:")
print(result.summary)

print("\nKey findings:")
for finding in result.key_findings:
    print("-", finding)

print("\nRecommended actions:")
for action in result.recommended_actions:
    print("-", action)

print("\nSupporting evidence count:")
print(len(result.supporting_evidence))

Case ID: CASE-002
Overall risk level: Critical
Confidence: Deterministic

Summary:
Case CASE-002 is classified as Critical priority based on 3 risk signals and 7 supporting evidence items.

Key findings:
- High aml risk signal from AML Model.
- High transaction anomaly signal from Anomaly Engine.
- High financial risk signal from Financial Risk Engine.
- Transaction amount exceeds the current account balance.
- Multiple login attempts were associated with the transaction.
- Customer has repeated delayed-payment behaviour.

Recommended actions:
- Escalate the case for immediate analyst review.
- Review recent transaction history for related unusual activity.
- Review authentication and login activity associated with the account.
- Review recent repayment behaviour and financial stress indicators.
- Review AML-related transaction evidence and linked account activity.

Supporting evidence count:
7


In [14]:
import importlib
import agents.investigation_agent.granite_client as granite_module

importlib.reload(granite_module)

granite_client = granite_module.GraniteClient()

missing_config = granite_client.validate_config()

if missing_config:
    print("Missing configuration:")
    for item in missing_config:
        print("-", item)
else:
    print("IBM Granite configuration detected successfully.")

Missing configuration:
- IBM_CLOUD_API_KEY
- IBM_PROJECT_ID
- IBM_WATSONX_URL


In [15]:
import importlib

import agents.investigation_agent.evidence as evidence_module
import agents.investigation_agent.investigator as investigator

importlib.reload(evidence_module)
importlib.reload(investigator)

# Fresh case
bob_test_case = investigator.create_investigation_case(
    case_id="CASE-BOB-001",
    customer_id="CUST-BOB-001",
    transaction_id="TX-BOB-001",
    aml_score=0.55,
    aml_level="Medium",
    anomaly_level="High",
    financial_risk_score=0.67,
    financial_risk_level="High"
)

# Attach signal evidence twice to test de-duplication
evidence_module.attach_signal_evidence(bob_test_case)
evidence_module.attach_signal_evidence(bob_test_case)

# Attach context evidence
evidence_module.attach_context_evidence(
    bob_test_case,
    {
        "TransactionAmount": 1500,
        "AccountBalance": 900,
        "LoginAttempts": 4,
        "DelayedPaymentMonths": 3,
        "Timestamp": "2026-08-18T14:30:00"
    }
)

print("Model signal evidence count:")
print(
    sum(
        1
        for item in bob_test_case.evidence
        if item.evidence_type == "Model Signal"
    )
)

print("\nStored context keys:")
print(
    [
        item.metadata.get("context_key")
        for item in bob_test_case.evidence
        if item.evidence_type == "Context"
    ]
)

findings = investigator.generate_findings(
    bob_test_case
)

actions = investigator.generate_recommended_actions(
    bob_test_case,
    findings
)

print("\nFindings:")
for finding in findings:
    print("-", finding)

print("\nRecommended actions:")
for action in actions:
    print("-", action)

Model signal evidence count:
3

Stored context keys:
['TransactionAmount', 'AccountBalance', 'LoginAttempts', 'DelayedPaymentMonths', 'Timestamp']

Findings:
- High transaction anomaly signal from Anomaly Engine.
- High financial risk signal from Financial Risk Engine.
- Transaction amount exceeds the current account balance.
- Multiple login attempts were associated with the transaction.
- Customer has repeated delayed-payment behaviour.

Recommended actions:
- Escalate the case for immediate analyst review.
- Review recent transaction history for related unusual activity.
- Review authentication and login activity associated with the account.
- Review recent repayment behaviour and financial stress indicators.
- Review AML-related transaction evidence and linked account activity.


In [16]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

print("API key loaded:", bool(os.getenv("IBM_CLOUD_API_KEY")))
print("Project ID loaded:", bool(os.getenv("IBM_PROJECT_ID")))
print("Watsonx URL loaded:", bool(os.getenv("IBM_WATSONX_URL")))

API key loaded: True
Project ID loaded: True
Watsonx URL loaded: True


In [18]:
from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

url = os.getenv("IBM_WATSONX_URL")

print("URL starts with https://:", url.startswith("https://"))
print("URL host:", url.replace("https://", ""))

URL starts with https://: True
URL host: ca-tor.ml.cloud.ibm.com


In [19]:
import os

from ibm_watsonx_ai import Credentials, APIClient

credentials = Credentials(
    url=os.getenv("IBM_WATSONX_URL"),
    api_key=os.getenv("IBM_CLOUD_API_KEY"),
)

watsonx_client = APIClient(
    credentials=credentials,
    project_id=os.getenv("IBM_PROJECT_ID"),
)

print("watsonx.ai client created successfully.")
print("Project ID configured:", bool(watsonx_client.default_project_id))

watsonx.ai client created successfully.
Project ID configured: True


In [20]:
print("Available chat models:")
watsonx_client.foundation_models.ChatModels.show()

Available chat models:
{'LLAMA_3_3_70B_INSTRUCT': 'meta-llama/llama-3-3-70b-instruct', 'MISTRAL_SMALL_3_1_24B_INSTRUCT_2503': 'mistralai/mistral-small-3-1-24b-instruct-2503'}


In [21]:
model_specs = watsonx_client.foundation_models.get_model_specs()

print("Granite models found:")
print("---------------------")

granite_models = []

for model in model_specs["resources"]:
    model_id = model.get("model_id", "")

    if "granite" in model_id.lower():
        granite_models.append(model_id)
        print(model_id)

print("\nTotal Granite models:", len(granite_models))

Granite models found:
---------------------

Total Granite models: 0


In [23]:
from agents.investigation_agent.watsonx_client import WatsonxClient

client = WatsonxClient()
model = client.get_primary_model()

response = model.generate_text(
    prompt="Reply with exactly: MonteCore watsonx integration successful."
)

print(response)

 If you are unable to do so, please explain why.

I'm unable to reply with "Mon


In [24]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a concise financial investigation assistant. "
            "Follow the user's formatting instructions exactly."
        ),
    },
    {
        "role": "user",
        "content": (
            "Reply with exactly this sentence and nothing else: "
            "MonteCore watsonx integration successful."
        ),
    },
]

chat_response = model.chat(
    messages=messages
)

print(
    chat_response["choices"][0]["message"]["content"]
)

MonteCore watsonx integration successful.


In [25]:
messages = [
    {
        "role": "system",
        "content": "You are a financial investigation assistant."
    },
    {
        "role": "user",
        "content": "Say hello in one short sentence."
    }
]

chat_response = model.chat(messages=messages)

print(chat_response["choices"][0]["message"]["content"])

Hello! How can I assist you with your financial investigation today?


In [26]:
import importlib

import agents.investigation_agent.watsonx_client as watsonx_module
import agents.investigation_agent.investigator as investigator

importlib.reload(watsonx_module)
importlib.reload(investigator)

deterministic_result = (
    investigator.build_deterministic_investigation_result(
        case_with_context
    )
)

watsonx_client = watsonx_module.WatsonxClient()

ai_summary = watsonx_client.generate_investigation_summary(
    case=case_with_context,
    deterministic_result=deterministic_result
)

print(ai_summary)

## Case Summary
The customer of this high-priority case shows multiple concurrent risk signals in AML, transaction anomaly, and  financial early warning categories. It should be reviewed immediately due to a significant discrepancy between transaction amount and account balance.

## Key Risk Indicators

- Multiple high-level risk signals in three distinct categories.
- High AML risk score of 0.820.
 - A significant and unexplained transaction amount versus the customer's account balance.
-  Frequent login attempts may indicate an attempt to access the account or disrupt monitoring

## Recommended Next Steps
- Examine the transaction related to the AML screening signal for suspicious patterns and particulate accounts.
- Review all recent transaction history and check for unusual velocity or amounts.
- Investigate the login attempts for potential gateway compromise.
  - Review any additional transactions and linked activity in affected accounts.

## Analyst Caution
This assessment is bas

In [27]:
import importlib

import agents.investigation_agent.watsonx_client as watsonx_module

importlib.reload(watsonx_module)

watsonx_client = watsonx_module.WatsonxClient()

review_output = watsonx_client.review_investigation_summary(
    case=case_with_context,
    deterministic_result=deterministic_result,
    primary_summary=ai_summary
)

print(review_output)

## Review Status
NEEDS_CORRECTION

## Unsupported or Overstated Claims
* "a significant discrepancy between transaction amount and account balance" - This statement implies a specific analysis of the transaction amount ($1531.42) and the account balance ($860.11) that is not present in the ground-truth evidence. While the numbers are provided, the evidence does not explicitly state that the discrepancy is significant or unexplained.
* "Frequent login attempts may indicate an attempt to access the account or disrupt monitoring" - The ground-truth evidence mentions "LoginAttempts: 4", but it does not provide any context or analysis suggesting that this is frequent or indicative of an attempt to access the account or disrupt monitoring.
* "Investigate the login attempts for potential gateway compromise" - The ground-truth evidence does not mention anything about a potential gateway compromise. This statement introduces an unsupported security-compromise claim.
* "particulate accounts" - T

In [28]:
import importlib

import agents.investigation_agent.orchestrator as orchestrator_module

importlib.reload(orchestrator_module)

agentic_result = orchestrator_module.run_agentic_investigation(
    case_with_context
)

print("Deterministic result available:",
      agentic_result.deterministic_result is not None)

print("Primary summary available:",
      agentic_result.primary_summary is not None)

print("Primary error:",
      agentic_result.primary_error)

print("Review output available:",
      agentic_result.review_output is not None)

print("Review error:",
      agentic_result.review_error)

Deterministic result available: True
Primary summary available: True
Primary error: None
Review output available: True
Review error: None


In [29]:
print("=== DETERMINISTIC SUMMARY ===")
print(agentic_result.deterministic_result.summary)

print("\n=== MISTRAL PRIMARY ASSESSMENT ===")
print(agentic_result.primary_summary)

print("\n=== LLAMA SECOND-LOOK REVIEW ===")
print(agentic_result.review_output)

=== DETERMINISTIC SUMMARY ===
Case CASE-002 is classified as Critical priority based on 3 risk signals and 7 supporting evidence items.

=== MISTRAL PRIMARY ASSESSMENT ===
## Case Summary
This case requires urgent review owing to high-priority risk signals indicating potential AML activity, transaction anomalies, and customer financial distress.

## Key Risk Indicators
- High Level AML Risk signal with score 0.820.
- High Level transaction anomaly signal.
- High Level Financial risk signal with score 0.670.
- Recent Transaction Amount 1531.42.
- Current Account Balance 860.11.
- Recent Login attempts 4.
- Delayed  Payment months 3

## Recommended Next Steps
- Immediately review recent transaction history for unusual activity.
- Assess AML-related transaction evidence and link to accounts.
- Conduct further investigation into the customer's financial situation, given the delayed payments

## Analyst Caution
This assessment is based on the provided data and models's signals; it does not 

In [30]:
from agents.investigation_agent.watsonx_client import WatsonxClient
from agents.investigation_agent.orchestrator import run_agentic_investigation

broken_client = WatsonxClient(
    api_key=None,
    project_id=None,
    url=None
)

fallback_result = run_agentic_investigation(
    case_with_context,
    watsonx_client=broken_client
)

print(
    "Deterministic result available:",
    fallback_result.deterministic_result is not None
)

print(
    "Primary summary available:",
    fallback_result.primary_summary is not None
)

print(
    "Primary error recorded:",
    fallback_result.primary_error is not None
)

print(
    "Review output available:",
    fallback_result.review_output is not None
)

print(
    "Review error recorded:",
    fallback_result.review_error is not None
)

print("\nPrimary error:")
print(fallback_result.primary_error)

print("\nReview error:")
print(fallback_result.review_error)

Deterministic result available: True
Primary summary available: True
Primary error recorded: False
Review output available: True
Review error recorded: False

Primary error:
None

Review error:
None


In [31]:
from agents.investigation_agent.watsonx_client import WatsonxClient
from agents.investigation_agent.orchestrator import run_agentic_investigation

broken_client = WatsonxClient()

# Deliberately invalidate this test client only.
# Your actual .env file remains untouched.
broken_client.api_key = None
broken_client.project_id = None
broken_client.url = None

fallback_result = run_agentic_investigation(
    case_with_context,
    watsonx_client=broken_client
)

print(
    "Deterministic result available:",
    fallback_result.deterministic_result is not None
)

print(
    "Primary summary available:",
    fallback_result.primary_summary is not None
)

print(
    "Primary error recorded:",
    fallback_result.primary_error is not None
)

print(
    "Review output available:",
    fallback_result.review_output is not None
)

print(
    "Review error recorded:",
    fallback_result.review_error is not None
)

print("\nPrimary error:")
print(fallback_result.primary_error)

print("\nReview error:")
print(fallback_result.review_error)

Deterministic result available: True
Primary summary available: False
Primary error recorded: True
Review output available: False
Review error recorded: True

Primary error:
Primary AI summarization unavailable: ValueError: Missing watsonx credentials: IBM_CLOUD_API_KEY, IBM_PROJECT_ID, IBM_WATSONX_URL

Review error:
Second-look review skipped: primary summary was not produced.
